# QF 627 Programming and Computational Finance
## Lesson 06 | Supervised Machine Learning for Predicting Stock Returns

> Hi Team 👋

> The cross-disciplinary field of quantitative and computational finance is not something that can be mastered simply by reading books or following random programming tutorials. The most effective approach is learning by doing, supported by systematic guidance and real-world case studies. This hands-on method is one of the best ways to develop both understanding and expertise in computational finance.

> Today, we’ll begin exploring machine learning through supervised learning for empirical asset pricing. As a practical example, we’ll work on predicting Microsoft’s stock price.

## DEPENDENCIES

In [1]:
# Load libraries.

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl

import seaborn as sns

import time

import datetime as dt
import re

# import plotly.express as px

import pandas_datareader.data as web
from pandas_datareader import data as pdr

import yfinance as yf

import warnings
warnings.filterwarnings("ignore")

# Setting baseline seed
np.random.seed(2025)

# Set print options.

np.set_printoptions(precision = 3)

plt.style.use("ggplot") # Grammar of Graphics Theme

mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.alpha"] = 0.25

mpl.rcParams["axes.facecolor"] = "white"

mpl.rcParams["legend.fontsize"] = 14

%matplotlib inline

In [2]:
%whos

Variable   Type      Data/Info
------------------------------
dt         module    <module 'datetime' from '<...>/python3.12/datetime.py'>
mpl        module    <module 'matplotlib' from<...>/matplotlib/__init__.py'>
np         module    Shape: <function shape at 0x103cf1080>
pd         module    <module 'pandas' from '/U<...>ages/pandas/__init__.py'>
pdr        module    <module 'pandas_datareade<...>ndas_datareader/data.py'>
plt        module    <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
re         module    <module 're' from '/Libra<...>thon3.12/re/__init__.py'>
sns        module    <module 'seaborn' from '/<...>ges/seaborn/__init__.py'>
time       module    <module 'time' (built-in)>
warnings   module    <module 'warnings' from '<...>/python3.12/warnings.py'>
web        module    <module 'pandas_datareade<...>ndas_datareader/data.py'>
yf         module    <module 'yfinance' from '<...>es/yfinance/__init__.py'>


## 👉 <a id = "top">Learning Pointers</a> 👈 

## [1. Problem Statement and Dependencies for Supervised ML](#p1)

> ### <font color = red> Define and Import </font>

## [2. Wrangle & Prepare Input Features](#p2)

> ### <font color = red> Feature Engineering </font>

## [3. Supervised Machine Learning: Step-by-Step](#p3)

> ### <font color = red> There are five steps </font>

## [4. A First Look at Hyperparameter Tuning](#p4)

> ### <font color = red> Tune `p`,`d`,`q` in ARIMA </font>

## [5. What We Have Learned Thus Far...](#p5)

> ### <font color = red> Supervised Learning for Price Prediction </font>


## <a id = "p1">1. </a> <font color = "green"> Problem Statement & Dependencies for Supervised ML </font>  [back to table of contents](#top)

> We will create a predictive model for the weekly return of MSFT stock. 

> To solve the problem, it is essential to understand what affects Microsoft’s stock price and to incorporate as much information as we can into the model. 

> Among correlated assets, technical indicators, and fundamental analysis, we will concentrate on correlated assets as features here.

### Activate Necessary Packages

> For the supervised regression models

In [3]:
# Step 3: Model Specification

In [4]:
# Let's import our built-in algorithms from scipy toolkit for machine learning

# Oldies But Goodies...

from sklearn.linear_model import LinearRegression # Least Squares

from sklearn.svm import SVR # Support Vector Machine

from sklearn.neighbors import KNeighborsRegressor # K-Nearest Neighbors

# Regularization (Penalized Regressors) --> Linear

from sklearn.linear_model import ElasticNet # Elastic Net Penalty
from sklearn.linear_model import Lasso # LASSO

# Decision Tree with Ensemble --> Non-linear

from sklearn.tree import DecisionTreeRegressor # Decision Tree

## Bagging (Bootstrapped-AGGregation) --> intelligent architecture like Homo Sapiens

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

## Boosting (more recent approaches) --> Homo Sapiens Sapiens

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor

In [5]:
## Neural Network (with less computational constraints)

## Multi-layer Perceptron

from sklearn.neural_network import MLPRegressor

In [7]:
# %whos

> For data analysis and model evaluation

In [8]:
### Step 1. Data Split

from sklearn.model_selection import train_test_split

In [9]:
### Step 2. Feature Engineering (Big 5 + 3)

from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import chi2

In [10]:
### Step 4. Hyper-parameter Tuning

from sklearn.model_selection import cross_val_score, KFold, GridSearchCV

> For time-series models for `machine learning`

In [11]:
import statsmodels.tsa.arima.model as stats

In [12]:
from statsmodels.graphics.tsaplots import plot_acf

> For data preparation and visualization

In [13]:
from pandas.plotting import scatter_matrix

### IMPORT Data

> shift + ctrl + `-` split the given cell

> In addition to historical data on Microsoft, the independent variables used are the following potentially correlated assets.

- Stocks
IBM (IBM) and Alphabet (GOOGL)

In [14]:
stock_ticker = ["MSFT", "IBM", "GOOGL"]

In [15]:
stock_data =\
(
    yf
    .download(stock_ticker)
    ["Close"]
)

[*********************100%***********************]  3 of 3 completed


In [16]:
stock_data

Ticker,GOOGL,IBM,MSFT
Date,,,
2025-09-02,211.161148,241.500000,505.119995
2025-09-03,230.453888,244.100006,505.350006
2025-09-04,232.092422,247.179993,507.970001
2025-09-05,234.790009,248.529999,495.000000
2025-09-08,234.039993,256.089996,498.200012
2025-09-09,239.630005,259.109985,498.410004
2025-09-10,239.169998,256.880005,500.369995
2025-09-11,240.369995,257.010010,501.010010
2025-09-12,240.800003,253.440002,509.899994


- Currency
USD/JPY and GBP/USD

In [17]:
forex_ticker = ["DEXJPUS", "DEXUSUK"]

In [18]:
forex_data =\
(
    pdr
    .get_data_fred(forex_ticker)
)

In [19]:
forex_data

,DEXJPUS,DEXUSUK
DATE,,
2020-10-01,105.53,1.2901
2020-10-02,105.36,1.2928
2020-10-05,105.70,1.2972
2020-10-06,105.64,1.2945
2020-10-07,105.97,1.2914
...,...,...
2025-09-22,147.83,1.3505
2025-09-23,147.83,1.3518
2025-09-24,148.71,1.3450


- Indices
S&P 500, Dow Jones, and VIX

In [20]:
index_ticker = ["SP500", "DJIA", "VIXCLS"]

In [21]:
index_data =\
(
    pdr
    .get_data_fred(index_ticker)
)

In [22]:
index_data

,SP500,DJIA,VIXCLS
DATE,,,
2020-10-01,3380.80,27816.90,26.70
2020-10-02,3348.44,27682.81,27.63
2020-10-05,3408.63,28148.64,27.96
2020-10-06,3360.95,27772.76,29.48
2020-10-07,3419.45,28303.46,28.06
...,...,...,...
2025-09-23,6656.92,46292.78,16.64
2025-09-24,6637.97,46121.28,16.18
2025-09-25,6604.72,45947.32,16.74


In [23]:
%whos

Variable                    Type         Data/Info
--------------------------------------------------
AdaBoostRegressor           ABCMeta      

TypeError: object of type 'ABCMeta' has no len()

## <a id = "p2">2. </a> <font color = "green"> Wrangle and Prepare Input Features </font>  [back to table of contents](#top)

### Data Transformation

> Next, we define our outcome (Y) and predictor (X) variables. The outcome variable is the weekly return of MSFT. The predicted variable is the weekly return of Microsoft (MSFT). 

> The number of trading days in a week is assumed to be five, and we compute the return using five trading days. For predictor variables we use the correlated assets and the historical return of MSFT at different frequencies.

> The variables used as predictors are lagged five-day returns of stocks (IBM and GOOG), currency exchange rates (USD/JPY and GBP/USD), and indices (S&P 500, Dow Jones, and VIX), along with lagged five-day, 15-day, 30-day, and 60-day returns of MSFT.

> The lagged five-day variables embed the time series component by using a time-delay approach, where the lagged variable is included as one of the predictor variables. This step translates the time series data into a supervised regression-based model framework.

### Outcome (Y): Lograthimc Returns of 5-day MSFT

In [ ]:
return_period = 5

In [ ]:
stock_data

In [ ]:
stock_data.loc[ : , "MSFT"]

In [ ]:
stock_data.loc[ : , "MSFT"].tail(5)

In [ ]:
stock_data.loc[ : , "MSFT"].tail(5).shift(1)

In [ ]:
Y =\
(
    np
    .log(stock_data.loc[ : , "MSFT"] # log return of MSFT
        )
    .diff(return_period)
    .shift(-return_period)
)

Y

In [ ]:
Y.name

In [ ]:
Y.name =\
(
    Y
    .name
    + 
    "_pred"
)

In [ ]:
Y

### Input Features (Xs)

In [ ]:
return_period

In [ ]:
X1 =\
(
    np
    .log(stock_data.loc[ : , ("IBM", "GOOGL")]
        )
    .diff(return_period)
)

X1

In [ ]:
X2 =\
(
    np
    .log(forex_data)
    .diff(return_period)
)

X2.head(6)

In [ ]:
X3 =\
(
    np
    .log(index_data)
    .diff(return_period)
)

In [ ]:
X3.head(6)

In [ ]:
Y.head(6)

> Team, FYI, after the logarithmic transformation, 
> the .diff(return_period) function is called on the resulting DataFrame. 

> This calculates the difference of consecutive log values (log-return) with a specified lag equal to return_period. 

> The return_period is a parameter representing the number of periods (rows) over which the difference is calculated.

In [ ]:
X4 =\
pd.concat(
    [np
    .log(stock_data.loc[ : , "MSFT"]
        )
    .diff(i) for i in [return_period,
                       return_period * 3, # 15
                       return_period * 6, # 30
                       return_period * 12 # 60
                      ]
    ],
    axis = 1
).dropna()

In [ ]:
X4.columns = ["MSFT_DT", "MSFT_3DT", "MSFT_6DT", "MSFT_12DT"]

In [ ]:
X4

> For each value i in the list [return_period, return_period * 3, return_period * 6, return_period * 12], 
> let's calculates the logarithm of the values in column ("Adj Close", "MSFT") 
> of DataFrame stock_data, 
> and then calculates the difference of consecutive log values (log-return) with lag i. 

> This results in a list of DataFrames.


> The DataFrames from the list comprehension are concatenated along the columns (axis=1),
> creating a single DataFrame. 

> After concatenation, rows containing any NaN values are dropped using .dropna().

In [ ]:
# all the input features 

X =\
(
    pd
    .concat([X1, X2, X3, X4],
            axis = 1)
)

X

In [ ]:
X.columns

> Let's concatenate two DataFrames Y and X along the columns, 
> dropping any rows with missing values, 
> and then selecting rows 
> at regular intervals specified by return_period. 

In [ ]:
data =\
(
    pd
    .concat([Y, X],
            axis = 1)
    .dropna()
)

In [ ]:
data.shape

In [ ]:
data.iloc[ : :return_period, : ].shape

In [ ]:
data =\
(
    data
    .iloc[ : :return_period, : ]
)

In [ ]:
data

In [ ]:
Y =\
(
    data
    .loc[ : , Y.name]
)

Y

In [ ]:
X.columns

In [ ]:
X =\
(
    data
    .loc[ : , X.columns]
)

In [ ]:
X

### Exploratory Data Analysis (EDA)

In [ ]:
correlation =\
(
    data
    .corr()
)

In [ ]:
(
    sns
    .heatmap(correlation,
             cmap = "viridis",
             annot = True)
)

In [ ]:
scatter_matrix(data,
               figsize = [16,16]
              )

plt.show()

## <a id = "p3">3. </a> <font color = "green"> Supervised Machine Learning: A Step-by-Step Guide </font>  [back to table of contents](#top)

### Step 1: Data Split

> Let's partition the original dataset into a training set and a test set.

In [ ]:
len(X)

In [ ]:
testing_set = 0.20

train_size = int(len(X) * (1 - testing_set)
                )

In [ ]:
train_size

In [ ]:
len(Y) == len(X) # cross-validate the data

In [ ]:
# Outcome Split

In [ ]:
Y_train, Y_test =\
(
    Y[0         : train_size],
    Y[train_size:len(Y)     ]
)

In [ ]:
# Input Features (Predictors) Split

In [ ]:
X_train, X_test =\
(
    X[0         : train_size],
    X[train_size:len(X)     ]
)

In [ ]:
len(X_train) 

In [ ]:
len(X_test)

In [ ]:
len(X)

### Preprations for Step 4: Ten-fold Cross Validation and Evaluation Metrics

In [ ]:
seed = 2025
num_folds = 10

metric = "neg_mean_squared_error"

### Step 3: Fitting: Model Comparison with ML Algorithms

In [ ]:
models = [] # staging

In [ ]:
sample_list = ["M", "Q", "F"]

In [ ]:
sample_list.append("627")

In [ ]:
sample_list

#### Regression and tree regression algorithms

# Least Squares

In [ ]:
(
    models
    .append(
        ("LR", LinearRegression()
         )
           )
)

# Regularization Algorithms 

In [ ]:
(
    models
    .append(
        ("LASSO", Lasso()
         )
           )
)

(
    models
    .append(
        ("Elastic Net Penalty", ElasticNet()
         )
           )
)

# Decision Tree (Greedy Algorithm)

In [ ]:
(
    models
    .append(
        ("Decision Tree", DecisionTreeRegressor()
         )
           )
)

# Ensemble models

In [ ]:
## Bagging 

(
    models
    .append(
        ("Random Forest", RandomForestRegressor()
         )
           )
)

(
    models
    .append(
        ("Extra Trees Algo", ExtraTreesRegressor()
         )
           )
)

In [ ]:
## Boosting

(
    models
    .append(
        ("Gradient Boosting", GradientBoostingRegressor()
         )
           )
)

(
    models
    .append(
        ("Adaptive Boosting", AdaBoostRegressor()
         )
           )
)

# Oldies But Goodies beyond Least Squares

In [ ]:
(
    models
    .append(
        ("Support Vector Machine", SVR()
         )
           )
)

(
    models
    .append(
        ("K-Nearest Neighbors", KNeighborsRegressor()
         )
           )
)

In [ ]:
len(models)

In [ ]:
models

In [ ]:
# models.pop(1)

In [ ]:
# models

In [ ]:
# len(models)

> Once you have selected all the models, you might want to loop over each of them. 

> First, let’s run the k-fold analysis. Then try to run the model on the entire training and testing dataset.

> All the algorithms use default tuning parameters. We will calculate the mean and standard deviation of the evaluation metric for each algorithm, collecting the results for model comparison.

### Best Practice in ML: Compare Algorithms' Performance

In [ ]:
# Let's build empty lists to store performance across many algorithms

names = []

train_results = []
test_results = []

kfold_results = []

In [ ]:
models

In [ ]:
num_folds

In [ ]:
metric

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
for name, model in models:

    names.append(name)

    # set k-fold cross-validation (here, 10)

    kfold =\
    (
        KFold(n_splits = num_folds,
              random_state = seed,
              shuffle = True)
    )

    # run cross-validation

    cv_results =\
    (
        -1
        *
        cross_val_score(model, X_train, Y_train,
                        cv = kfold,
                        scoring = metric)
    )

    # Cross-validation results should be contained in objects

    kfold_results.append(cv_results)

    # Training Set Model Fitting

    res = model.fit(X_train, Y_train)

    # Assess the performance in Training Set

    train_result = mean_squared_error(res.predict(X_train), Y_train)

    train_results.append(train_result)

    # Assess the performance in Testing Set

    test_result = mean_squared_error(res.predict(X_test), Y_test)

    test_results.append(test_result)    

    user_interface_message = "%s: %f (%f) %f %f " % (name, 
                                                     cv_results.mean(), 
                                                     cv_results.std(), 
                                                     train_result, 
                                                     test_result)

    print(user_interface_message)

#### <mark>Comparing the algorithms by examining the Training vs. Testing Set</mark>

In [ ]:
names

In [ ]:
names

In [ ]:
len(names)

In [ ]:
names * 2

In [ ]:
len(train_results)

In [ ]:
len(test_results)

In [ ]:
df_for_comparison =\
(
    pd
    .DataFrame(
        {"Algorithms": names * 2,
         "Data": ["Training Set"] * len(names) + ["Testing Set"] * len(names),
         "Performance": train_results + test_results
         }
    )
)

In [ ]:
df_for_comparison

In [ ]:
df_for_comparison.columns

In [ ]:
from lets_plot import *
LetsPlot.setup_html()

In [ ]:
performance_comparison =\
(
    ggplot(df_for_comparison,
           aes(x = "Algorithms",
               y = "Performance",
               fill = "Data"
              )
          )
    + geom_bar(stat = "identity",
               position = "dodge",
               width = 0.5)
    + labs(title = "Comparing the Performance of Machine Learning Algorithms on the Training vs. Testing Set",
           y = "Mean Squared Error (MSE)",
           x = "Name of ML Algorithms",
           caption = "Source: Federal Reserve Bank & Yahoo Finance")
    + theme(legend_position = "top")
    + ggsize(1000, 500)
)

performance_comparison.show()

### Time Series based models (ARIMA)

> Let us first prepare the dataset for ARIMA models, by having only the correlated variables as exogenous variables.

#### ARIMA

In [ ]:
X_train.columns

In [ ]:
X_train_ARIMA =\
(
    X_train
    .loc[ : , ['IBM', 'GOOGL', 'DEXJPUS', 'DEXUSUK', 'SP500', 'DJIA', 'VIXCLS']]
)

X_test_ARIMA =\
(
    X_test
    .loc[ : , ['IBM', 'GOOGL', 'DEXJPUS', 'DEXUSUK', 'SP500', 'DJIA', 'VIXCLS']]
)

In [ ]:
len(X_train_ARIMA) + len(X_test_ARIMA) == len(X)


* `p` denotes the order of Auto Regression (AR) polynomials

* `d` denotes the number of nonseasonal differences needed for stationarity

* `q` denotes the order of Moving Average (MA) polynomials


In [ ]:
baseline_ARIMA =\
(
    stats
    .ARIMA(endog = Y_train,
           exog = X_train_ARIMA,
           order = [1, 0, 0]
          )
)

In [ ]:
ARIMA_fitted =\
(
    baseline_ARIMA
    .fit()
)

In [ ]:
error_training_ARIMA =\
(
    mean_squared_error(Y_train, 
                       ARIMA_fitted.fittedvalues)
)

In [ ]:
predicted =\
(
    ARIMA_fitted
    .predict(start = len(X_train_ARIMA) - 1,
             end = len(X) - 1,
             exog = X_test_ARIMA
            )[1 : ]
)

In [ ]:
predicted

In [ ]:
error_testing_ARIMA =\
(
    mean_squared_error(Y_test,
                       predicted)
)

In [ ]:
error_training_ARIMA

In [ ]:
error_testing_ARIMA

## <a id = "p4">4. </a> <font color = "green"> An Introduction to Hyperparameter Tuning </font>  [back to table of contents](#top)

### <mark>A Gift</mark> Model Tuning and Grid Search for ARIMA

In [ ]:
# Hyperparameter Tuning; Grid Search for ARIMA

def assess_ARIMA_model(arima_order):
    
    modelARIMA = stats.ARIMA(endog = Y_train, 
                             exog = X_train_ARIMA,
                             order = arima_order)
    # Our model takes an arima_order as input, 
    # fits an ARIMA model to the training data Y_train 
    # with exogenous variables X_train_ARIMA, 
    
    model_fit = modelARIMA.fit()
    # and then calculates 

    error = mean_squared_error(Y_train,
                               model_fit.fittedvalues)
    
    # and returns the Mean Squared Error (MSE) 
    # between the true and the fitted values.

    return error

def assess_models(p_values, d_values, q_values):
    
    # Team, our function performs grid search 
    # over all combinations of provided p, d, and q values. 
    
    # For each combination, it calculates the MSE and prints it. 
    
    # If the MSE for the current combination 
    # is less than the best score encountered so far, 
    # it updates the best score and the corresponding configuration. 
    
    # At the end of the grid search, 
    # it prints the best configuration and its MSE.
    
    best_score, best_cfg = float("inf"), None

    for p in p_values:
        for d in d_values:
            for q in q_values:
                order = (p, d, q)
                try:
                    mse = assess_ARIMA_model(order)
                    if mse < best_score:
                        best_score, best_cfg = mse, order
                    
                    print("ARIMA%s MSE = %.7f" % (order, mse)
                          )
                    
                except:
                    continue
    print("Best ARIMA%s MSE = %.7f" % (best_cfg, best_score)
          )
    
# parameters to use for assessment

# Recall that the ARIMA model 
# is characterized by three parameters: 
# (p, d, q) which stand for the order of autoregression, 
# the degree of differencing, 
# and the order of the moving average, respectively.

p_values = [0, 1, 2]
d_values = range(0, 2)
q_values = range(0, 2)

> Calculate accuracy on testing data

In [ ]:
assess_models(p_values,
              d_values,
              q_values)

In [ ]:
ARIMA_Tuned =\
(
    stats
    .ARIMA(endog = Y_train,
           exog = X_train_ARIMA,
           order = [2, 0, 0]
          )
)

ARIMA_algo_tuned = ARIMA_Tuned.fit()

In [ ]:
predictions_ARIMA_TUNED =\
(
    ARIMA_algo_tuned
    .predict(start = len(X_train_ARIMA) - 1,
             end = len(X) - 1,
             exog = X_test_ARIMA
            )[1 : ]
)

- `model_fit` is a fitted ARIMA model instance.
<br>

- `model_fit.predict()` is used to make predictions based on the fitted model.
<br>

- `start=train_len - 1` specifies the starting point of the prediction. Our code is setting it to one less than the length of the training data.
<br>

- `end=total_len - 1` sets the endpoint of the prediction. It is one less than the total length of the dataset, indicating that predictions will be made for the entire testing set.
<br>

- `exog=X_test_ARIMA` refers to exogenous variables, which are external factors that can influence the predictions. X_test_ARIMA is presumably the testing set of these exogenous variables.
<br>

- `[1:]` is used to exclude the first element from the predicted values. It might be done to align the predicted values with the actual values, especially if the prediction includes the last observation from the training set.

### Use `lets_plot` for ***interactive*** <mark>dashboarding-like</mark> reporting

In [ ]:
predictions_ARIMA_TUNED.index = Y_test.index

In [ ]:
predictions_ARIMA_TUNED

In [ ]:
actual_results = np.exp(Y_test).cumprod() # y ==> true future prices

In [ ]:
pred_results = np.exp(predictions_ARIMA_TUNED).cumprod() # ==> predicted future prices 

In [ ]:
difference = actual_results - pred_results

In [ ]:
df_for_prediction_outcomes =\
(
    pd
    .DataFrame(
        {"date": actual_results.index,
         "future MSFT": actual_results.values,
         "predicted MSFT": pred_results.values,
         "difference": difference.values}
    )
)

In [ ]:
df_for_ggplot =\
(
    df_for_prediction_outcomes
    .melt(id_vars = ["date", "difference"],
          var_name = "series",
          value_name = "value")
)

In [ ]:
df_for_ggplot

In [ ]:
ARIMA_based_asset_pricing =\
(
    ggplot(df_for_ggplot,
           aes(x = "date",
               y = "value",
               color = "series")
          )
    + geom_line()
    + geom_point()
    + scale_y_continuous(limits = [0, 2]
                        )
    + scale_color_manual(
        {
            "future MSFT": "blue",
            "predicted MSFT": "red"
        }
                         )
    + labs(title = "Predicting MSFT Stock Returns with Supervised Machine Learning",
           x = "Date",
           y = "Cumulative Returns",
           color = "series")
    + theme(legend_position = "top")
    + ggsize(1000, 500)
)

In [ ]:
ARIMA_based_asset_pricing.show()

In [ ]:
df_for_ggplot.columns

In [ ]:
tooltips =\
(
    layer_tooltips()
        .format("value", ".2f")
        .format("difference", ".2f")
        .line("date: @{date}")
        .line("series: @{series}")
        .line("value: @{value}")
        .line("difference: @{difference}")
)

In [ ]:
tooltips

In [ ]:
ARIMA_based_asset_pricing_with_tooltips =\
(
    ggplot(df_for_ggplot,
           aes(x = "date",
               y = "value",
               color = "series")
          )
    + geom_line()
    + geom_point(tooltips = tooltips)
    + scale_y_continuous(limits = [0, 2]
                        )
    + scale_color_manual(
        {
            "future MSFT": "blue",
            "predicted MSFT": "red"
        }
                         )
    + labs(title = "Predicting MSFT Stock Returns with Supervised Machine Learning",
           x = "Date",
           y = "Cumulative Returns",
           color = "series")
    + theme(legend_position = "top")
    + ggsize(1000, 500)
)

In [ ]:
ARIMA_based_asset_pricing_with_tooltips.show()

## <a id = "p3">5. </a> <font color = "green"> What We Have Learned Thus Far... </font>  [back to table of contents](#top)

### What We Learned from Machine Learning

> It appears that simple models such as linear regression, regularized regression (i.e., Lasso and elastic net), along with time series models such as ARIMA, are promising modeling approaches for asset price prediction problems.

> Such algorithms could help financial practitioners to model time dependencies in a more flexible way. 

> In our problem-solving machine learning lesson, you have learned how to address overfitting and underfitting, which are among the key challenges in prediction problems in computational finance. Do please note that you could use a better set of indicators, such as P/E ratio, trading volume, technical indicators, or news data. To do so might lead to better results (and we will indeed make use of such indicators in following lessons).

> `Thank you for working with the script, Team 👍`